<a href="https://colab.research.google.com/github/rajkdas/Unlimited-OCR/blob/main/OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages for Unlimited-OCR
!pip install -q accelerate einops addict easydict pymupdf psutil gradio safetensors

In [ ]:
import sys
import transformers.utils.import_utils as import_utils
import transformers.utils as utils
import sentencepiece as spm

# ---- Patch 1: missing import_utils helpers (removed in transformers v5) ----
missing_utils = [
    'is_torch_fx_available',
    'is_flash_attn_2_available',
    'is_flash_attn_available',
    'is_torch_sdpa_available',
]
for util_name in missing_utils:
    if not hasattr(import_utils, util_name):
        setattr(import_utils, util_name, lambda: False)
    if not hasattr(utils, util_name):
        setattr(utils, util_name, lambda: False)

# ---- Patch 2: LlamaFlashAttention2 (removed in transformers 4.47+) ----
try:
    from transformers.models.llama import modeling_llama
    if not hasattr(modeling_llama, "LlamaFlashAttention2"):
        class LlamaFlashAttention2(modeling_llama.LlamaAttention):
            pass
        modeling_llama.LlamaFlashAttention2 = LlamaFlashAttention2
except Exception:
    pass

# ---- Patch 3: SentencePiece Load (tokenizer.model is missing in the repo) ----
original_sp_load = spm.SentencePieceProcessor.Load
def patched_sp_load(self, model_file=None, model_proto=None):
    if model_file is None and model_proto is None:
        return
    return original_sp_load(self, model_file, model_proto)
spm.SentencePieceProcessor.Load = patched_sp_load

print("✅ Successfully patched transformers and sentencepiece for v5.x compatibility!")

# ---- Load tokenizer ----
import os
import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig

print("Loading Unlimited-OCR...")
model_name = 'baidu/Unlimited-OCR'

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True, use_fast=True)

# ---- Load config and fix missing special-token-id fields ----
config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)

# transformers v5 raises AttributeError for token-id fields not present in config.json.
# Set them explicitly on the instance. (Deliberately NOT adding a broad __getattr__ —
# that breaks hasattr() checks used internally by transformers and caused the
# deepcopy / heterogeneity errors.)
for _attr in ("pad_token_id", "bos_token_id", "eos_token_id",
              "sep_token_id", "unk_token_id", "mask_token_id"):
    if not hasattr(config, _attr):
        object.__setattr__(config, _attr, None)

print("✅ Config patched.")

# ---- Load model using the patched config ----
model = AutoModel.from_pretrained(
    model_name,
    config=config,
    trust_remote_code=True,
    use_safetensors=True,
    torch_dtype='auto',
)
model = model.eval().cuda()
print("✅ Model loaded successfully on GPU!")

✅ Successfully patched transformers and sentencepiece for v5.x compatibility!
Loading Unlimited-OCR...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Config patched.


`torch_dtype` is deprecated! Use `dtype` instead!
Some weights of UnlimitedOCRForCausalLM were not initialized from the model checkpoint at baidu/Unlimited-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded successfully on GPU!


In [ ]:
import os
import pymupdf  # Updated from 'fitz' to 'pymupdf'
import tempfile

def pdf_to_images(pdf_path, dpi=300):
    """Converts PDF pages into high-res PNG images."""
    doc = pymupdf.open(pdf_path)
    tmp_dir = tempfile.mkdtemp(prefix='pdf_ocr_')
    mat = pymupdf.Matrix(dpi / 72, dpi / 72)
    paths = []
    for i, page in enumerate(doc):
        out = os.path.join(tmp_dir, f'page_{i+1:04d}.png')
        page.get_pixmap(matrix=mat).save(out)
        paths.append(out)
    doc.close()
    return paths

print("✅ Helper functions loaded successfully!")

✅ Helper functions loaded successfully!


In [ ]:
import gradio as gr
import shutil

def extract_text_from_pdf(pdf_filepath):
    if pdf_filepath is None:
        return "Please upload a PDF file."

    # 1. Convert PDF to images
    image_paths = pdf_to_images(pdf_filepath, dpi=300)
    output_dir = tempfile.mkdtemp(prefix='pdf_output_')

    try:
        # 2. Run multi-page parsing (Base mode for PDFs)
        outputs, output_tokens = model.infer_multi(
            tokenizer,
            prompt='<image>Multi page parsing.',
            image_files=image_paths,
            output_path=output_dir,
            image_size=1024,         # Recommended for multi-page PDFs
            max_length=32768,        # Full context window for long docs
            no_repeat_ngram_size=35, # Max accuracy setting for PDFs
            ngram_window=1024,       # Max accuracy setting for PDFs
            save_results=True,       # Triggers internal text/bbox cleanup
        )

        # 3. Read the cleaned markdown file generated by the model
        result_md_path = os.path.join(output_dir, 'result.md')
        if os.path.exists(result_md_path):
            with open(result_md_path, 'r', encoding='utf-8') as f:
                clean_outputs = f.read()
        else:
            clean_outputs = outputs # Fallback to raw output

        # 4. Clean up tags for a more readable text view
        clean_outputs = clean_outputs.replace("<PAGE>", "\n\n--- Page Break ---\n\n")
        clean_outputs = clean_outputs.replace("<PAGE>\n", "\n\n--- Page Break ---\n\n")

        # 5. Clean up temporary files to save Colab disk space
        shutil.rmtree(output_dir, ignore_errors=True)
        shutil.rmtree(os.path.dirname(image_paths[0]), ignore_errors=True)

        return clean_outputs

    except Exception as e:
        return f"Error during extraction: {str(e)}"

# Create the dynamic Gradio interface
iface = gr.Interface(
    fn=extract_text_from_pdf,
    inputs=gr.File(label="Upload a PDF file", file_types=[".pdf"], type="filepath"),
    outputs=gr.Textbox(label="Extracted Text", lines=25), # Removed 'show_copy_button=True' for compatibility
    title="Unlimited-OCR PDF Extractor",
    description="Upload a PDF document to extract text with max accuracy using Baidu's Unlimited-OCR. (Note: Very large PDFs may take a few minutes to process)."
)

# Launch the app
# share=True creates a public URL you can open in a new tab
iface.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://6d5befa83a64ee90fb.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.13/dist-packages/gradio/routes.py:1368: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<PAGE><|det|>title [115, 97, 478, 130]<|/det|>Performance of OCR
<|det|>text [114, 135, 346, 152]<|/det|>Author: Roger Dunham
<|det|>text [114, 163, 327, 180]<|/det|>Date: 8 $ ^{th} $ July 2019
<|det|>title [114, 227, 325, 248]<|/det|>Executive Summary
<|det|>text [112, 251, 871, 287]<|/det|>SolidOCR was compared with Acrobat and ReadIRIS to evaluate the quality and speed of reconstructing PDFs into editable Word Documents.
<|det|>text [112, 299, 858, 334]<|/det|>All three options resulted in generally high quality reconstruction, though some character recognition errors occurred in each case.
<|det|>text [113, 346, 609, 362]<|/det|>In all cases, serif fonts were slower to identify than sans-serif fonts.
<|det|>text [112, 374, 844, 409]<|/det|>SolidOCR offered significantly faster reconstruction per page, and the differential between serif and sans-serif was considerably smaller than for the other OCR options.
<|det|>title [114, 456, 229, 476]<|/det|>Objectives
<|det|>text [112, 480, 8

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 20/20 [00:00<00:00, 87018.76it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_1: 100%|██████████| 13/13 [00:00<00:00, 63773.04it/s]
image_page_2: 0it [00:00, ?it/s]
other_page_3: 100%|██████████| 9/9 [00:00<00:00, 22878.02it/s]
image_page_4: 0it [00:00, ?it/s]
other_page_4: 100%|██████████| 1/1 [00:00<00:00, 10433.59it/s]
image_page_5: 0it [00:00, ?it/s]
other_page_5: 100%|██████████| 2/2 [00:00<00:00, 29959.31it/s]
image_page_6: 0it [00:00, ?it/s]
other_page_6: 100%|██████████| 15/15 [00:00<00:00, 82456.83it/s]
image_page_7: 0it [00:00, ?it/s]
other_page_7: 100%|██████████| 12/12 [00:00<00:00, 20493.34it/s]
image_page_8: 0it [00:00, ?it/s]
other_page_8: 100%|██████████| 4/4 [00:00<00:00, 54827.50it/s]
image_page_9: 0it [00:00, ?it/s]
other_page_9: 100%|██████████| 13/13 [00:00<00:00, 53352.20it/s]
image_page_10: 0it [00:00, ?it/s]
other_page_10: 100%|██████████| 12/12 [00:00<00:00, 95144.89it/s]
image_page_11: 0it [00:00,

invalid character '，' (U+FF0C) (<string>, line 1)
invalid character '，' (U+FF0C) (<string>, line 1)
closing parenthesis '}' does not match opening parenthesis '[' (<string>, line 1)
'{' was never closed (<string>, line 1)
'{' was never closed (<string>, line 1)
invalid character '，' (U+FF0C) (<string>, line 1)


image_page_12: 0it [00:00, ?it/s]
other_page_12: 100%|██████████| 199/199 [00:00<00:00, 96862.77it/s]
